In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import LabelEncoder
import json
pd.options.display.max_columns = None

In [55]:
data_path = 'data/'
df_lists = []

customers_df = pd.read_csv(data_path + 'olist_customers_dataset.csv')
geolocation_df = pd.read_csv(data_path + 'olist_geolocation_dataset.csv')
order_items_df = pd.read_csv(data_path + 'olist_order_items_dataset.csv')
order_payments_df = pd.read_csv(data_path + 'olist_order_payments_dataset.csv')
order_reviews_df = pd.read_csv(data_path + 'olist_order_reviews_dataset.csv')
orders_df = pd.read_csv(data_path + 'olist_orders_dataset.csv')
products_df = pd.read_csv(data_path + 'olist_products_dataset.csv')
sellers_df = pd.read_csv(data_path + 'olist_sellers_dataset.csv')
category_translation_df = pd.read_csv(data_path + 'product_category_name_translation.csv')

df_lists.append(customers_df)
df_lists.append(geolocation_df)
df_lists.append(order_items_df)
df_lists.append(order_payments_df)
df_lists.append(order_reviews_df)
df_lists.append(orders_df)
df_lists.append(products_df)
df_lists.append(sellers_df)
df_lists.append(category_translation_df)

df_names = ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']

### **Arreglar olist_order_reviews_dataset.csv**

In [56]:
# Para arreglar los saltos de líneas en los comentarios se hará lo siguiente:
# Como sabemos que todas las filas tienen si o si un review_answer_timestamp, de no encontrar ese campo en una fila se concatenará con la siguiente fila hasta encontrarlo.
fixed_rows = []
i = 0

while i < len(order_reviews_df):
    row = order_reviews_df.iloc[i].copy()
    
    # Mientras la fila actual NO tenga review_answer_timestamp
    while pd.isna(row['review_answer_timestamp']) and (i + 1) < len(order_reviews_df):
        # Concatenamos con la siguiente fila
        next_row = order_reviews_df.iloc[i + 1]

        # Concatenamos campo por campo
        for col in order_reviews_df.columns:
            if pd.isna(row[col]) or row[col] == '':
                row[col] = next_row[col]
            else:
                if not pd.isna(next_row[col]) and next_row[col] != '':
                    row[col] = str(row[col]) + " " + str(next_row[col])

        i += 1

    fixed_rows.append(row)
    i += 1

order_reviews_df = pd.DataFrame(fixed_rows)


### **Creación del DataFrame**

In [58]:
# 1. Merge Customers + Orders
orders_customers_df = orders_df.merge(customers_df, on='customer_id', how='left')

# 2. Merge Orders + Reviews
orders_reviews_df = orders_customers_df.merge(order_reviews_df[['order_id', 'review_score']], on='order_id', how='left')

# 3. Total spent y Payment details
payments_agg = order_payments_df.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_type': lambda x: x.mode()[0],  # modo (más frecuente)
    'payment_installments': 'max'            # número máximo de cuotas (debería ser una sola si solo hay un pago)
}).reset_index()

orders_reviews_payments_df = orders_reviews_df.merge(payments_agg, on='order_id', how='left')

# 4. Freight value, Price Mean, Products Info
order_items_agg = order_items_df.groupby('order_id').agg({
    'freight_value': 'sum',
    'price': 'mean'
}).rename(columns={
    'price': 'price_mean_per_order'
}).reset_index()

orders_full = orders_reviews_payments_df.merge(order_items_agg, on='order_id', how='left')

# 5. Delivery Delay
orders_full['order_delivered_customer_date'] = pd.to_datetime(orders_full['order_delivered_customer_date'])
orders_full['order_estimated_delivery_date'] = pd.to_datetime(orders_full['order_estimated_delivery_date'])

orders_full['delivery_delay'] = (orders_full['order_delivered_customer_date'] - orders_full['order_estimated_delivery_date']).dt.days

# 6. Purchased category frequency (categoría más frecuente por pedido)
order_items_products_df = order_items_df.merge(products_df[['product_id', 'product_category_name']], on='product_id', how='left')

# La categoría más frecuente
category_freq = order_items_products_df.groupby('order_id')['product_category_name']\
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)\
    .reset_index()\
    .rename(columns={'product_category_name': 'purchased_category_frequency'})

orders_full = orders_full.merge(category_freq, on='order_id', how='left')

# 7. Highest value category (categoría del producto más caro por pedido)
order_items_products_df['rank'] = order_items_products_df.groupby('order_id')['price'].rank(method='first', ascending=False)

highest_category = order_items_products_df[order_items_products_df['rank'] == 1][['order_id', 'product_category_name']]\
    .rename(columns={'product_category_name': 'highest_category_value'})

orders_full = orders_full.merge(highest_category, on='order_id', how='left')

orders_full.rename(columns={'payment_value': 'total_spent'}, inplace=True)

# 8. Calcular order_size
order_size = order_items_df.groupby('order_id')['order_item_id'].count().reset_index().rename(columns={'order_item_id': 'order_size'})


orders_full = orders_full.merge(order_size, on='order_id', how='left')

# 9. Selección de columnas
df_customer_value = orders_full[[
    'customer_unique_id',
    'total_spent',
    'order_size',
    'review_score',
    'customer_state',
    'purchased_category_frequency',
    'highest_category_value',
    'delivery_delay',
    'payment_type',
    'payment_installments',
    'price_mean_per_order',
    'freight_value'
]].copy()

### **Imputar valores nulos o faltantes**

In [59]:
# Definir columnas numéricas y categóricas
numeric_cols = ['total_spent', 'order_size', 'review_score', 'delivery_delay', 
                'payment_installments', 'price_mean_per_order', 'freight_value']

categorical_cols = ['customer_state', 'purchased_category_frequency', 'highest_category_value', 'payment_type']

# Imputar NaN en numéricos con la mediana
for col in numeric_cols:
    if col in df_customer_value.columns:
        median_value = df_customer_value[col].median()
        df_customer_value[col] = df_customer_value[col].fillna(median_value)


### **Codificación de Variables Categóricas**

In [60]:
def apply_label_encoding(df, column, return_mapping=False):
    if column not in df.columns or df[column].isnull().all():
        if return_mapping:
            return df, None
        return df

    le = LabelEncoder()
    df_copy = df.copy()

    temp_col = df_copy[column].fillna('MISSING')
    le.fit(temp_col)
    df_copy[column] = le.transform(temp_col)

    if return_mapping:
        mapping = dict(zip(le.classes_, range(len(le.classes_))))
        return df_copy, mapping

    return df_copy


# Función  para one-hot encoding
def apply_onehot_encoding(df, column, max_categories=15):
    if column not in df.columns:
        return df

    df_copy = df.copy()
    n_unique = df_copy[column].nunique()

    if n_unique <= max_categories:
        dummies = pd.get_dummies(df_copy[column], prefix=column, dummy_na=df_copy[column].isnull().any())
        df_copy = pd.concat([df_copy, dummies], axis=1)
        df_copy.drop(column, axis=1, inplace=True)
    
    return df_copy

In [61]:
encoding_mappings = {}
for col in categorical_cols:
    df_customer_value, mapping = apply_label_encoding(df_customer_value, col, return_mapping=True)
    encoding_mappings[col] = mapping

with open('encoding_mappingsV2.json', 'w', encoding='utf-8') as f:
    json.dump(encoding_mappings, f, ensure_ascii=False, indent=4)


### **Escalar Variables**

In [62]:
from sklearn.preprocessing import StandardScaler

# Columnas a escalar
cols_to_scale = ['total_spent', 'delivery_delay', 'payment_installments', 'price_mean_per_order', 'freight_value']

scaler = StandardScaler()

df_customer_value_scaled = df_customer_value.copy()
df_customer_value_scaled[cols_to_scale] = scaler.fit_transform(df_customer_value_scaled[cols_to_scale])
